# 04 - Explainability & Fairness Audit
## InclusionScore AI - Alternate Credit Scoring

This notebook generates:
1. **Global SHAP** summary (bar + beeswarm plots)
2. **Local SHAP** waterfall explanations for 5 individual samples
3. **Fairness audit** with subgroup AUCs, disparate impact, and demographic parity

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from src.explainability import compute_shap_summary, local_explanation, generate_local_explanations
from src.fairness import run_audit
from src.models import load_model
from src.features import load_processed

%matplotlib inline

## 1. Global SHAP Explainability

Compute SHAP values on a sample of the test set and plot global feature importance.

In [ ]:
MODEL_PATH = '../models/xgb_v1.joblib'
DATA_PATH = '../data/processed/test.parquet'

shap_values = compute_shap_summary(MODEL_PATH, DATA_PATH)
print(f'SHAP values shape: {shap_values.shape}')

### SHAP Summary Bar Plot
![SHAP Summary](../reports/explainability/shap_summary.png)

### SHAP Beeswarm Plot
![SHAP Beeswarm](../reports/explainability/shap_beeswarm.png)

## 2. Local SHAP Explanations

Generate waterfall plots for 5 individual test samples, showing which features push the prediction up or down.

In [ ]:
local_results = generate_local_explanations(MODEL_PATH, DATA_PATH, indices=[0, 1, 2, 3, 4])

for i, r in enumerate(local_results):
    print(f"\nSample {i}: predicted default prob = {r['prediction']:.3f}")
    print(f"  Top 3 contributing features:")
    for feat in r['top_features'][:3]:
        direction = '+' if feat['contribution'] > 0 else ''
        print(f"    {feat['feature']:40s} value={feat['value']:>10.3f}  contribution={direction}{feat['contribution']:.4f}")

### Local Explanation Plots

![Local 0](../reports/explainability/shap_local_0.png)
![Local 1](../reports/explainability/shap_local_1.png)
![Local 2](../reports/explainability/shap_local_2.png)
![Local 3](../reports/explainability/shap_local_3.png)
![Local 4](../reports/explainability/shap_local_4.png)

## 3. Fairness Audit

Run the fairness audit using age-based subgroups (since the dataset has no gender/ethnicity/religion columns).

In [ ]:
audit_results = run_audit(MODEL_PATH, DATA_PATH)

print('\n=== Subgroup Metrics ===')
print(pd.DataFrame(audit_results['subgroup_metrics']).to_string(index=False))

print('\n=== Disparate Impact ===')
di = audit_results['disparate_impact']
print(f"Best group:  {di['best_group']} (approval rate {di['best_approval_rate']:.2%})")
print(f"Worst group: {di['worst_group']} (approval rate {di['worst_approval_rate']:.2%})")
print(f"DI ratio:    {di['di_ratio']:.4f}  {'PASS' if di['pass'] else 'FAIL'}")

print('\n=== Demographic Parity ===')
dp = audit_results['demographic_parity']
print(f"Parity gap:  {dp['parity_gap']:.4f}  {'PASS' if dp['pass'] else 'FAIL'}")

### Subgroup AUC Chart
![Subgroup AUC](../reports/explainability/subgroup_auc.png)

## 4. Summary

- **Global SHAP**: `RevolvingUtilizationOfUnsecuredLines`, `age`, and delinquency counts are the most important features.
- **Local SHAP**: Waterfall plots show clear, interpretable explanations per applicant.
- **Fairness**: Age-based disparate impact ratio is **0.44** (below 0.80 threshold) -- younger borrowers have significantly lower approval rates. This is expected given the strong age-default correlation in the data.
- **Mitigation**: Per-group threshold calibration or post-processing adjustments recommended for production deployment.

Full reports saved to:
- `reports/fairness_report.md`
- `reports/explainability/*.png`